# Lab Assessment — Alternative B

Rewritten version with alternative answers and analysis wording.


In [ ]:
# SaaS Customer Churn Lab — Rewritten Solution
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="ticks")
plt.rcParams["figure.figsize"] = (9, 5)

np.random.seed(42)
N = 1200

tenure = np.random.exponential(scale=18, size=N).clip(1, 72).astype(int)
contract = np.random.choice(
    ["Month-to-Month", "One-Year", "Two-Year"], size=N,
    p=[0.55, 0.25, 0.20]
)
support_tickets = np.random.poisson(lam=1.8, size=N) + (
    contract == "Month-to-Month"
) * np.random.poisson(lam=1.2, size=N)
monthly_charges = np.random.normal(75.0, 25.0, size=N).clip(20.0, 150.0)
tech_support = np.random.choice(
    ["Yes", "No", "No internet"], size=N,
    p=[0.40, 0.45, 0.15]
)
payment_method = np.random.choice(
    ["Electronic Check", "Bank Transfer", "Credit Card"], size=N,
    p=[0.40, 0.30, 0.30]
)

churn_logits = (
    -2.0 + 0.03 * monthly_charges - 0.05 * tenure
    + 0.45 * support_tickets
    + 1.2 * (contract == "Month-to-Month")
    - 1.0 * (contract == "Two-Year")
    + 0.8 * (payment_method == "Electronic Check")
)
churn_prob = 1 / (1 + np.exp(-churn_logits))
churn = np.where(np.random.rand(N) < churn_prob, "Yes", "No")

total_charges = tenure * monthly_charges + np.random.normal(0, 50, size=N)
total_charges[np.random.choice(N, size=15, replace=False)] = np.nan

df = pd.DataFrame({
    "Customer_ID": [f"CUST-{2000+i}" for i in range(N)],
    "Tenure_Months": tenure,
    "Contract_Type": contract,
    "Payment_Method": payment_method,
    "Tech_Support": tech_support,
    "Support_Tickets": support_tickets,
    "Monthly_Charges": np.round(monthly_charges, 2),
    "Total_Charges": np.round(total_charges, 2),
    "Churn": churn
})
df.to_csv("saas_churn_data.csv", index=False)
print("Dataset created:", df.shape)


In [ ]:
# TASK 1.1 — Alternative Tenure Analysis
plt.figure(figsize=(8, 4))
sns.boxenplot(data=df, x="Churn", y="Tenure_Months")
plt.title("Customer Tenure by Churn Outcome")
plt.xlabel("Customer churn")
plt.ylabel("Months active")
plt.tight_layout()
plt.show()

stats_alt = df.groupby("Churn")["Tenure_Months"].agg(
    median="median",
    q25=lambda x: x.quantile(.25),
    q75=lambda x: x.quantile(.75)
)
stats_alt["IQR"] = stats_alt["q75"] - stats_alt["q25"]
display(stats_alt.round(2))

# Answer:
# The distribution for churned customers is shifted toward shorter tenure,
# suggesting that churn prevention should start well before renewal decisions.


In [ ]:
# TASK 1.2 — Alternative Segment Table
segment = (
    df.groupby(["Contract_Type", "Tech_Support", "Churn"])
      .size()
      .groupby(level=[0, 1])
      .transform(lambda x: x / x.sum() * 100)
      .rename("Percent")
      .reset_index()
)

segment_table = segment.pivot_table(
    index=["Contract_Type", "Tech_Support"],
    columns="Churn", values="Percent", fill_value=0
)
display(segment_table.round(2))

segment_table.plot(kind="bar", figsize=(11, 5))
plt.title("Churn Composition of Contract/Support Segments")
plt.ylabel("Percent of segment")
plt.xlabel("Segment")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Answer:
# The table reinforces that contract duration is an important segmentation
# variable. Technical support changes the picture within some contract groups,
# so the two variables are best considered together.


In [ ]:
# TASK 1.3 — Alternative Statistical Test
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df, x="Monthly_Charges", y="Total_Charges",
    hue="Churn", alpha=.55
)
plt.title("Recurring Charges and Cumulative Customer Spend")
plt.tight_layout()
plt.show()

pairs = df[["Monthly_Charges", "Total_Charges"]].dropna()
pearson_test = stats.pearsonr(pairs["Monthly_Charges"], pairs["Total_Charges"])
spearman_test = stats.spearmanr(pairs["Monthly_Charges"], pairs["Total_Charges"])

print(f"Pearson r = {pearson_test.statistic:.3f}; p = {pearson_test.pvalue:.3g}")
print(f"Spearman rho = {spearman_test.statistic:.3f}; p = {spearman_test.pvalue:.3g}")

# Answer:
# The positive relationship is statistically strong in this generated data.
# The association is also expected because total charges incorporate the
# recurring monthly charge over the customer's tenure.


In [ ]:
# TASK 2.1 — Alternative Heatmap
df_num = df.assign(Churn_Flag=(df["Churn"] == "Yes").astype(int))
numeric = df_num.select_dtypes(include="number")

plt.figure(figsize=(10, 7))
sns.heatmap(
    numeric.corr(), annot=True, fmt=".2f",
    cmap="coolwarm", center=0, linewidths=.3
)
plt.title("Numeric Correlation Heatmap")
plt.tight_layout()
plt.show()

# Answer:
# The heatmap is most useful as a screening tool: it highlights which numeric
# variables deserve closer investigation, rather than establishing causality.


In [ ]:
# TASK 2.2 — Alternative Segmentation View
g = sns.FacetGrid(
    df, col="Contract_Type", hue="Churn",
    height=4, col_wrap=3
)
g.map_dataframe(
    sns.scatterplot,
    x="Monthly_Charges", y="Support_Tickets", alpha=.7
)
g.add_legend()
g.set_axis_labels("Monthly charges", "Support tickets")
g.set_titles("{col_name}")
plt.show()

# Answer:
# Contract-specific panels reveal that support-ticket behavior should be
# interpreted in the context of the contract category rather than globally.


In [ ]:
# TASK 3.1 — Alternative Imputation
missing = df["Total_Charges"].isna()
replacement = df.loc[missing, "Tenure_Months"] * df.loc[missing, "Monthly_Charges"]
df.loc[missing, "Total_Charges"] = replacement

print("Imputed records:", int(missing.sum()))
print("Remaining missing values:", int(df["Total_Charges"].isna().sum()))

# Answer:
# Formula-based imputation preserves the individual customer's expected spend
# structure, whereas a single mean or median would erase customer-level differences.


In [ ]:
# TASK 3.2 — Alternative Outlier Check
q1 = df["Support_Tickets"].quantile(.25)
q3 = df["Support_Tickets"].quantile(.75)
iqr = q3 - q1
cutoff = q3 + 1.5 * iqr

high_ticket = df.loc[df["Support_Tickets"] > cutoff]
print(f"Outlier cutoff: {cutoff:.2f}")
print(f"High-ticket accounts: {len(high_ticket)}")
print(f"Churn rate in this group: {high_ticket['Churn'].eq('Yes').mean()*100:.2f}%")

# Answer:
# The outlier group can be used as a priority-review list. It should not be
# treated as a deterministic churn class.


In [ ]:
# TASK 4 — Alternative Feature Set
df["Ticket_Velocity"] = df["Support_Tickets"] / (df["Tenure_Months"] + 1)

# Financial-risk indicator based on the upper quartile.
price_q3 = df["Monthly_Charges"].quantile(.75)
df["Premium_Price_Flag"] = (df["Monthly_Charges"] >= price_q3).astype(int)

# Original assignment risk definition retained for comparison.
df["High_Risk_Flag"] = (
    (df["Contract_Type"] == "Month-to-Month") &
    (df["Tech_Support"] == "No")
).astype(int)

print("Premium price flag:")
display(
    df.groupby("Premium_Price_Flag")["Churn"]
      .value_counts(normalize=True).unstack().mul(100).round(2)
)

print("Ticket velocity:")
display(
    df.groupby("Churn")["Ticket_Velocity"]
      .agg(["median", "mean", "max"]).round(3)
)

# Answer:
# The two added features capture different risks: support pressure relative to
# tenure and exposure to a relatively high recurring price.


# TASK 6 — Strategic Insights

### Insight 1 — Longer commitments are associated with better retention
- **Claim:** Moving customers from flexible contracts toward longer commitments is likely to have the greatest broad retention impact.
- **Evidence:** Churn percentages vary strongly across the contract categories in the normalized table.
- **Business Action:** Present annual plans as a value upgrade rather than waiting until cancellation is imminent.

### Insight 2 — Ticket velocity adds context to raw support counts
- **Claim:** A burst of support requests from a newer customer is more concerning than the same count spread across a long tenure.
- **Evidence:** `Ticket_Velocity` explicitly scales support activity by tenure.
- **Business Action:** Trigger customer-success outreach when ticket velocity becomes unusually high.

### Insight 3 — Onboarding is a strategic retention activity
- **Claim:** Shorter-tenure customers deserve disproportionate attention.
- **Evidence:** The tenure distributions place churned customers at a lower typical tenure than retained customers.
- **Business Action:** Introduce milestone-based onboarding and follow-up after unresolved support cases.

### Insight 4 — Pricing and lifetime value should be analyzed together
- **Claim:** A high monthly price does not by itself describe total customer value or churn exposure.
- **Evidence:** Total charges are positively related to monthly charges while also depending on tenure.
- **Business Action:** Segment offers using both recurring price and customer age instead of applying one blanket discount.

### Insight 5 — Targeted retention beats universal discounts
- **Claim:** Different customer groups require different interventions.
- **Evidence:** Contract/support combinations and the multidimensional plots show heterogeneous customer segments.
- **Business Action:** Use contract-upgrade offers for flexible plans, service recovery for support-heavy accounts, and onboarding help for newer customers.


In [ ]:
# TASK 7 — Submission Checklist
# 1. Run the notebook from the first cell to the last.
# 2. Download the completed .ipynb from Colab.
# 3. Add the notebook to the portfolio repository.
# 4. Commit and push the work, for example:
#    git add .
#    git commit -m "complete rewritten SaaS churn EDA"
#    git push origin main
# 5. Confirm the README includes findings, methodology, visualizations,
#    and recommended strategic actions.
